# Pyrates FEP
# Notebook 03: Coding grid pretest

## 1. Game selection

### Inputs:
- Interaction traces: `data/input/cleaned/cleaned_interaction_traces.csv`
- Experts for this pilot: Penny, Shaquille

Targets students who reached at least level 5 but never reached level 7 (i.e. did not complete level 6), a plateau point worth a closer look. About 20 feedback instances per group (B/C), trying for at least one instance from levels 1, 3, 4 and 5 in each group, and preferring games with at least 5 feedback instances so the sample is concentrated into fewer sessions rather than spread one-per-game.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root -> data/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "input" / "cleaned" / "cleaned_interaction_traces.csv"

print("Project root :", PROJECT_ROOT.name)
print("Input        :", INPUT_PATH.relative_to(PROJECT_ROOT), "| exists:", INPUT_PATH.exists())

traces = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, low_memory=False)
feedback = traces[traces["object_name"] == "ASSISTANT_HELP"]

In [ ]:
PLATEAU_TARGET_LEVELS = [1, 3, 4, 5]
PLATEAU_TARGET_PER_GROUP = 20
PLATEAU_MIN_FEEDBACK_PER_GAME = 5  # preferred: games with at least this many feedback instances

level_id_int = traces["level_id"].astype(int)
max_level_reached = level_id_int.groupby(traces["game_id"]).max()
plateau_games = max_level_reached[(max_level_reached >= 5) & (max_level_reached <= 6)].index

plateau_feedback = feedback[
    (feedback["action_name"] == "RECEIVED") & (feedback["game_id"].isin(plateau_games))
].copy()
plateau_feedback["level_id"] = plateau_feedback["level_id"].astype(int)

print(f"Games that reached level 5 but not level 7: {len(plateau_games)}")
print(f"Feedback instances available in those games: {len(plateau_feedback)}")
print(plateau_feedback.groupby(["group_id", "level_id"]).size().rename("feedback").to_frame())

In [ ]:
def select_plateau_sample(group_df, target_n, target_levels, min_per_game=5, seed=0):
    # shuffle once (deterministic tie-break), so game order below is a fixed random order
    shuffled = group_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    by_game = {game_id: rows["id"].tolist() for game_id, rows in shuffled.groupby("game_id")}
    level_of = dict(zip(shuffled["id"], shuffled["level_id"]))

    # games with >= min_per_game feedback instances are preferred, smallest of them first, so
    # the target is approached closely rather than overshot by a single very large session;
    # games below the threshold are only a fallback if the preferred games can't reach the target
    game_sizes = shuffled.groupby("game_id").size()
    big_games = list(game_sizes[game_sizes >= min_per_game].sort_values(kind="stable").index)
    small_games = [g for g in game_sizes.sort_values(kind="stable").index if g not in set(big_games)]

    def has_level(game_id, level):
        return any(level_of[fid] == level for fid in by_game[game_id])

    # Step 1: which games must be included to guarantee at least one instance per target level?
    # every game ends up included *in full* (see step 2), so a level already covered by an
    # already-required game needs no extra game
    required_games: list[str] = []
    for level in target_levels:
        if any(has_level(g, level) for g in required_games):
            continue
        for game_id in big_games + small_games:
            if has_level(game_id, level):
                required_games.append(game_id)
                break

    # Step 2: add whole games - never a partial game - smallest-first among big games, until
    # their combined feedback reaches the target; a game is either fully in or fully out
    pool = list(required_games)
    total = sum(len(by_game[g]) for g in pool)
    for game_pool in (big_games, small_games):
        for game_id in game_pool:
            if total >= target_n:
                break
            if game_id not in pool:
                pool.append(game_id)
                total += len(by_game[game_id])
        if total >= target_n:
            break

    selected_ids = [feedback_id for game_id in pool for feedback_id in by_game[game_id]]
    return group_df.set_index("id").loc[selected_ids].reset_index()


plateau_sample = pd.concat(
    [
        select_plateau_sample(
            plateau_feedback[plateau_feedback.group_id == group],
            PLATEAU_TARGET_PER_GROUP,
            PLATEAU_TARGET_LEVELS,
            min_per_game=PLATEAU_MIN_FEEDBACK_PER_GAME,
        )
        for group in ["B", "C"]
    ],
    ignore_index=True,
).sort_values(["group_id", "game_id", "level_id"]).reset_index(drop=True)

print(f"Selected {len(plateau_sample)} feedback instances "
      f"(B={int((plateau_sample.group_id == 'B').sum())}, C={int((plateau_sample.group_id == 'C').sum())}) "
      f"- target is {PLATEAU_TARGET_PER_GROUP}/group, but every selected game is included in full, "
      f"so the actual total can run a little over")
print(f"Distinct games represented: {plateau_sample['game_id'].nunique()}")
print("Feedback instances per selected game:")
print(plateau_sample.groupby(["group_id", "game_id"]).size().rename("selected").reset_index().groupby("group_id")["selected"].describe())

print("\nLevel coverage (selected instances):")
print(plateau_sample.groupby(["group_id", "level_id"]).size().rename("selected").to_frame())

for level in PLATEAU_TARGET_LEVELS:
    for group in ["B", "C"]:
        covered = ((plateau_sample.group_id == group) & (plateau_sample.level_id == level)).any()
        if not covered:
            print(f"NOTE: no level {level} feedback available for group {group} in this sample.")

# every selected game_id should carry its *complete* feedback set - not a subset of it
coverage_check = (
    plateau_feedback.groupby("game_id").size().rename("available")
    .to_frame()
    .join(plateau_sample.groupby("game_id").size().rename("selected"), how="right")
)
assert (coverage_check["selected"] == coverage_check["available"]).all(), coverage_check[coverage_check["selected"] != coverage_check["available"]]
print("\nOK: every selected game's feedback is included in full (no partial games).")

plateau_sample[["id", "date", "game_id", "level_id", "group_id", "student_id"]]

In [ ]:
PLATEAU_OUTPUT_PATH = DATA_DIR / "output" / "plateau_feedback_sample.csv"
plateau_sample.to_csv(PLATEAU_OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
print(f"Written: {PLATEAU_OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({len(plateau_sample)} rows)")

In [ ]:
pilot_games = pd.read_csv(PLATEAU_OUTPUT_PATH, dtype=str)["game_id"].unique()
pilot_game_list = ";".join(sorted(pilot_games))
print(f"Games selected for the pilot ({len(pilot_games)}):")
print(pilot_game_list)

### Output:
The distinct `game_id` values in that sample are the games assigned to the pilot, as one semicolon-separated list:
- `6iDc21P;CVtd56h;J6davsE;SPCYvuq;UF66dCx;X2TaH4S;b4cAb8f;sWWsfut`

## 2. Coding on the FEP platform

### Inputs:
- Game list from part 1 (2 experts: Penny, Shaquille)
- Coding form: `campaign/1_pilot/pilot_form.json`
- Cleaned trace dataset: `data/input/cleaned/cleaned_interaction_traces.csv`

The two experts code the selected games on the FEP platform, which lets them replay each game in full. No code here: the coding itself happens on the platform, outside this notebook.

### Output:
`campaign/1_pilot/pilot_result.csv`, one row per (game, trace, expert) with the expert's answers to the form.

## 3. Results analysis

### Inputs:
- `campaign/1_pilot/pilot_result.csv`

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Paths relative to the notebook: notebooks/ -> project root -> campaign/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "campaign" / "1_pilot" / "pilot_result.csv"
EXPERT_COL = "expert_id"
GAME_COL = "game_id"
TRACE_COL = "trace_id"
EXPERTS = ["Penny", "Shaquille"]

SINGLE_LABEL_AXES = ["resolution_stage"]
MULTI_LABEL_AXES = ["feedback_invalidity", "feedback_content", "feedback_instruction"]

print("Project root :", PROJECT_ROOT.name)
print("Data path    :", DATA_PATH.relative_to(PROJECT_ROOT), "| exists:", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)

In [ ]:
def norm_set(value):
    if not value:
        return frozenset()
    return frozenset(v.strip() for v in value.split(","))


def get_pairs(df, col):
    pivot = df.pivot_table(index=[GAME_COL, TRACE_COL], columns=EXPERT_COL, values=col, aggfunc="first")
    return pivot[EXPERTS[0]], pivot[EXPERTS[1]]


def jaccard_distance(a, b):
    if not a and not b:
        return 0.0
    return 1 - len(a & b) / len(a | b)


def masi_distance(a, b):
    if not a and not b:
        return 0.0
    if a == b:
        m = 1.0
    elif a <= b or b <= a:
        m = 2 / 3
    elif a & b:
        m = 1 / 3
    else:
        m = 0.0
    jaccard = len(a & b) / len(a | b) if (a | b) else 0.0
    return 1 - jaccard * m


def krippendorff_alpha(pairs, distance_fn):
    n = len(pairs)
    observed = sum(distance_fn(v1, v2) for v1, v2 in pairs) / n
    pooled = [v for pair in pairs for v in pair]
    total = len(pooled)
    expected_sum = sum(
        distance_fn(pooled[i], pooled[j]) for i in range(total) for j in range(total) if i != j
    )
    expected = expected_sum / (total * (total - 1))
    return float("nan") if expected == 0 else 1 - observed / expected


def cohen_kappa(values_a, values_b):
    """Wrapper kept so per-response tables read the same whichever axis they come from."""
    if values_a.nunique() == 1 and values_b.nunique() == 1 and values_a.iloc[0] == values_b.iloc[0]:
        return 1.0
    return cohen_kappa_score(values_a, values_b)

In [ ]:
def single_label_report(df, col):
    e1, e2 = get_pairs(df, col)
    n = len(e1)
    accord_brut = (e1 == e2).mean()
    kappa_total = cohen_kappa(e1, e2)
    categories = sorted(set(e1) | set(e2))
    rows = []
    for cat in categories:
        b1, b2 = e1 == cat, e2 == cat
        rows.append({
            "response": cat,
            "kappa": round(cohen_kappa(b1, b2), 2),
            "effectif": int((b1 | b2).sum()),
            f"{EXPERTS[0]}_n": int(b1.sum()),
            f"{EXPERTS[1]}_n": int(b2.sum()),
        })
    return {
        "n": n,
        "accord_brut": round(accord_brut, 3),
        "kappa_total": round(kappa_total, 2),
        "per_response": pd.DataFrame(rows),
    }


def multi_label_report(df, col):
    e1, e2 = get_pairs(df, col)
    n = len(e1)
    pairs = [(norm_set(a), norm_set(b)) for a, b in zip(e1, e2)]
    accord_brut = sum(a == b for a, b in pairs) / n
    alpha_jaccard = krippendorff_alpha(pairs, jaccard_distance)
    alpha_masi = krippendorff_alpha(pairs, masi_distance)
    labels = set()
    for v in df[col]:
        labels |= norm_set(v)
    rows = []
    for label in sorted(labels):
        b1 = e1.apply(lambda x: label in norm_set(x))
        b2 = e2.apply(lambda x: label in norm_set(x))
        rows.append({
            "response": label,
            "kappa": round(cohen_kappa(b1, b2), 2),
            "effectif": int((b1 | b2).sum()),
            f"{EXPERTS[0]}_n": int(b1.sum()),
            f"{EXPERTS[1]}_n": int(b2.sum()),
        })
    return {
        "n": n,
        "accord_brut": round(accord_brut, 3),
        "alpha_jaccard": round(alpha_jaccard, 2),
        "alpha_masi": round(alpha_masi, 2),
        "per_response": pd.DataFrame(rows),
    }


def binary_validity_report(df, col, valid_label="R0-NONE"):
    e1, e2 = get_pairs(df, col)

    def is_invalid(value):
        labels = norm_set(value)
        return bool(labels) and labels != {valid_label}

    b1 = e1.apply(is_invalid)
    b2 = e2.apply(is_invalid)
    return {
        "accord_brut": round((b1 == b2).mean(), 3),
        "kappa": round(cohen_kappa(b1, b2), 2),
        f"{EXPERTS[0]}_n_invalid": int(b1.sum()),
        f"{EXPERTS[1]}_n_invalid": int(b2.sum()),
    }

In [ ]:
reports = {}
for axis in SINGLE_LABEL_AXES:
    reports[axis] = single_label_report(df, axis)
for axis in MULTI_LABEL_AXES:
    reports[axis] = multi_label_report(df, axis)

for axis, report in reports.items():
    print(f"\n=== {axis} ===")
    print({k: v for k, v in report.items() if k != "per_response"})
    print(report["per_response"])

In [ ]:
binary_validity_report(df, "feedback_invalidity")

## 4. Summary

| Axis | Raw agreement | Kappa | Alpha (whole axis) | Status |
| --- | --- | --- | --- | --- |
| Resolution stage | 70% | 0.62 | 0.62 (same value, nominal axis) | Fair under Landis & Koch; below Krippendorff's strict threshold (0.667) |
| Invalidity | 84% | 0.13 to 0.23 (unbalanced categories) | 0.17 | Unreliable under both conventions, rare categories to monitor |
| Content | 25% | 0.06 to 0.86 depending on the label | 0.38 (Jaccard) / 0.31 (MASI) | Unreliable under both conventions; critical issue on C1-TASK / C4-PROC |
| Instructions | 70% | 0.48 to 1.00 depending on the label | 0.72 (Jaccard) / 0.66 (MASI) | Good under Landis & Koch; only tentative under Krippendorff's stricter criterion |

**Priority 1**: clarify the definition of C1-TASK and C4-PROC in the content grid before the full campaign. This is the only axis where the disagreement is systematic (Penny consistently more extensive than Shaquille: 2.36 vs 1.57 labels per trace on average) rather than due to chance or the small pilot sample.

**Priority 2**: clarify the S2-NAIV / S3-BLOC boundary on the resolution stage axis, which concentrates 7 of the 13 disagreements on that axis.

**To monitor, no immediate action needed**: the rare categories on the invalidity and instruction axes (R1-PYTH, R2-GAME, I3-EXEC, C6-EXAM), whose reliability cannot be assessed on a 44-trace sample.